For Replication of Figures from the Publication

In [9]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

In [ ]:
import textwrap

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.font_manager import FontProperties

from maintenance_llm.utils.paths import OUTPUT_DIR

output_dir = OUTPUT_DIR / "figures"
output_dir.mkdir(parents=True, exist_ok=True)

Preset Master Font

In [67]:
font = FontProperties(family="Calibri", size=16)

##### Class level Heatmap

In [ ]:
"""Heatmap figure Function"""

def plot_class_level_f1_score_heatmap(f1_score, title, output_figure_name):
    plt.figure(figsize=(14, 12))

    sns.heatmap(
        f1_score,
        vmin=0.0,
        vmax=1.0,
        xticklabels=f1_score.columns,
        yticklabels=[
            "blades",
            "control system",
            "electrical components",
            "gearbox",
            "generator",
            "grease, oil and coolant",
            "heaters and coolers",
            "hub",
            "others",
            "pitch and hydraulics",
            "power supply and converter",
            "pumps and motors",
            "safety",
            "scheduled",
            "sensors",
            "tower and foundation",
            "transformer",
            "yaw system",
        ],
        annot=True,
        annot_kws={
            "fontsize": 12,
        },
        cmap="Blues",
        linewidths=0.5,
        fmt=".2f",
    )


    plt.xticks(fontproperties=font)
    plt.yticks(fontproperties=font)
    plt.title(
        title,
        fontproperties=font,
        weight="bold",
    )

    # uncomment the following line to save the figure as a svg file
    # plt.savefig(
    #     output_dir / output_figure_name, bbox_inches="tight", dpi=300
    # )


Base Prompt Class Level F1-Score

In [ ]:
# load compiled class level f1 score data under base prompt configuration  
base_prompt_class_level_f1_score = pd.read_csv(f"{output_dir}/figures_data/base_prompt_class_level_f1_score.csv")

# Execute heatmap plotting function for base prompt configuration
plot_class_level_f1_score_heatmap(
    base_prompt_class_level_f1_score,
    "Base Prompt - Class Level F1-Scores",
    "class_performance_base_prompt.svg",
)

RAG Class Level F1-Score

In [ ]:
# load compiled class level f1 score data under RAG prompt configuration
RAG_class_level_f1_score = pd.read_csv(f"{output_dir}/figures_data/RAG_class_level_f1_score.csv")

# Execute heatmap plotting function for RAG prompt configuration
plot_class_level_f1_score_heatmap(
    RAG_class_level_f1_score,
    "RAG Prompt - Class Level F1-Scores",
    "class_performance_rag_prompt.svg",
)

##### Base Prompt vs RAG Configuration

In [58]:
# load compiled results for each metric 
base_vs_rag = pd.read_csv(f"{output_dir}/figures_data/base_vs_rag.csv")

In [ ]:
""" Generate Bar Plot for macro-F1-Scores of open-source LLMs """

open_source_macrof1 = base_vs_rag[base_vs_rag["Model"].isin(["Mistral-7B", "Qwen3.5-9B", "Llama3-8B", "Gemma3-4B"])]

plt.figure(figsize=(12, 8))

ax = sns.barplot(
    data=open_source_macrof1,
    x="Model",
    y="Macro F1-score",
    hue="prompt_config",
    palette="Blues",  # Clean, professional colour scheme
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=2,
        fontsize=14,
    )

ax.set_axisbelow(True)  # Ensure gridlines are below the bars

ax.grid(
    axis="y",  # gridlines along x-axis
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

ax.legend(
    title="Inference Method",
    fontsize=11,
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
)

plt.ylim(0, 0.8)

plt.xticks(fontproperties=font)
plt.yticks(fontproperties=font)
plt.xlabel("", fontsize=11)
plt.ylabel("Macro F1-Score", labelpad=15,fontsize=16)

plt.tight_layout()

# uncomment the following line to save the figure as a svg file
# plt.savefig(
#     output_dir / "RAG_effect.svg", bbox_inches="tight", dpi=300
#     )

In [ ]:
""" Generate Bar Plot comparing metric scores for [Qwen3.5-9B + RAG] and [GPT-5.6-luna + Base Prompt] """
comparison = base_vs_rag[base_vs_rag["Model"].isin(["Qwen3.5-9B", "GPT-5.6-luna"])]

plot_df = comparison.melt(
    id_vars = ["Model", "prompt_config"], 
    value_vars = [ "Accuracy", "Macro Precision", "Macro Recall", "Macro F1-score"],
    var_name = "Metric",
    value_name = "Metric Scores"
)
plot_df["Model+Config"] = plot_df["Model"] + " + " + plot_df["prompt_config"]

drop_config = [
    "Qwen3.5-9B + Base Prompt",
    "GPT-5.6-luna + RAG"
]

plot_df = plot_df[~plot_df["Model+Config"].isin(drop_config)]

plt.figure(figsize=(12, 6))


ax = sns.barplot(
    data=plot_df,
    y="Metric",
    x="Metric Scores",
    hue="Model+Config",
    palette="Reds",  # Clean, professional colour scheme
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=2,
        fontsize=9,
    )

ax.set_axisbelow(True)  # Ensure gridlines are below the bars

ax.grid(
    axis="x",  # gridlines along x-axis
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

handles, labels = ax.get_legend_handles_labels()

wrapped_labels = [textwrap.fill(label, width=15) for label in labels]

ax.legend(
    handles,
    wrapped_labels,
    title="",
    loc="upper right",
    bbox_to_anchor=(0.995, 0.85),
    # ncol=1,
    fontsize=10,
)
ax.set_yticklabels(["Accuracy", "Macro\nPrecision", "Macro\nRecall", "Macro\nF1-score"])

plt.xlim(0, 0.9)

plt.title("", fontsize=11)
plt.xticks(fontproperties=font)
plt.yticks(fontproperties=font)
plt.xlabel("Metric Score", labelpad=15, fontsize=14)
plt.ylabel(" ", fontsize=8)

plt.tight_layout()


# uncomment the following line to save the figure as a svg file
# plt.savefig(
    # output_dir / "RAG Qwen vs Base GPT.svg", bbox_inches="tight", dpi=300
    # )

In [ ]:
""" Generate Bar Plot comparing metric scores for [Qwen3.5-9B + RAG] and [GPT-5.6-luna + RAG] """
comparison = base_vs_rag[base_vs_rag["Model"].isin(["Qwen3.5-9B", "GPT-5.6-luna"])]

plot_df = comparison.melt(
    id_vars = ["Model", "prompt_config"], 
    value_vars = [ "Accuracy", "Macro Precision", "Macro Recall", "Macro F1-score"],
    var_name = "Metric",
    value_name = "Metric Scores"
)
plot_df["Model+Config"] = plot_df["Model"] + " + " + plot_df["prompt_config"]

drop_config = [
    "Qwen3.5-9B + Base Prompt",
    "GPT-5.6-luna + Base Prompt"
]

plot_df = plot_df[~plot_df["Model+Config"].isin(drop_config)]

plt.figure(figsize=(12, 6))


ax = sns.barplot(
    data=plot_df,
    y="Metric",
    x="Metric Scores",
    hue="Model+Config",
    palette="Reds",  # Clean, professional colour scheme
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=2,
        fontsize=9,
    )

ax.set_axisbelow(True)  # Ensure gridlines are below the bars

ax.grid(
    axis="x",  # gridlines along x-axis
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
)

handles, labels = ax.get_legend_handles_labels()

wrapped_labels = [textwrap.fill(label, width=15) for label in labels]

ax.legend(
    handles,
    wrapped_labels,
    title="",
    loc="upper right",
    bbox_to_anchor=(0.995, 0.85),
    # ncol=1,
    fontsize=10,
)
ax.set_yticklabels(["Accuracy", "Macro\nPrecision", "Macro\nRecall", "Macro\nF1-score"])

plt.xlim(0, 0.9)

plt.title("", fontsize=11)
plt.xticks(fontproperties=font)
plt.yticks(fontproperties=font)
plt.xlabel("Metric Score", labelpad=15, fontsize=14)
plt.ylabel(" ", fontsize=8)

plt.tight_layout()


# uncomment the following line to save the figure as a svg file
# plt.savefig(
#     output_dir / "RAG Qwen vs RAG GPT.svg", bbox_inches="tight", dpi=300
#     )